## 标准全连接层
标准全连接层的每个输出神经元都与所有输入神经元全连接，权重矩阵尺寸为 输出维度 × 输入维度，加上偏置项后，总参数量与输入、输出维度的乘积成正比，即参数复杂度为 $\mathcal{O}(d \cdot q)$。

本段代码通过 PyTorch 原生 nn.Linear 计算标准全连接层的真实参数量，作为后续 PHM 轻量化方案的对比基准。


In [ ]:
import torch
import torch.nn as nn

d = 512  # 输入维度
q = 512  # 输出维度
batch_size = 32

# 原生标准全连接层
standard_fc = nn.Linear(in_features=d, out_features=q, bias=True)

# 前向传播
x = torch.randn(batch_size, d)
out = standard_fc(x)

# 统计参数量
param_count = sum(p.numel() for p in standard_fc.parameters())
print(f"标准全连接总参数量：{param_count}")
# 输出：262656 = 512*512(权重) + 512(偏置)

## PHM 超复数全连接层
PHM（Parameterized Hypercomplex Multiplication，参数化超复数乘法）是一种基于代数结构的轻量化全连接方案，核心是通过克罗内克积（Kronecker Product）用小参数矩阵组装出大权重矩阵，在不改变输入输出维度的前提下，大幅压缩参数量。
核心实现逻辑
1.	设定超参数 $n$（超复数维度），要求输入维度 $d$ 和输出维度 $q$ 均可被 $n$ 整除
2.	定义两组可学习参数：
- 规则矩阵 $A$：形状为 $(n, n, n)$，共 $n$ 个 $n×n$ 小矩阵，用于学习超复数的乘法交互规则
- 基底矩阵 $S$：形状为 $(n, q/n, d/n)$，共 $n$ 个小尺寸特征变换矩阵

3.	前向传播时，通过 $H = \sum_{i=1}^n A_i \otimes S_i$（克罗内克积求和）组装出完整的 $q×d$ 权重矩阵，再执行标准线性变换

**参数量压缩效果**

总可学习参数量为 $n^3 + \frac{dq}{n}$。在深度学习典型场景下，特征维度远大于 $n$，$n^3$ 可忽略，参数量近似为标准全连接的 $1/n$，参数复杂度降至 $\mathcal{O}(\frac{dq}{n})$。

本段代码自定义实现 PHMLinear 层，并统计其实际参数量，与标准全连接层形成直观对比。


In [ ]:
import torch.nn.functional as F


class PHMLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, n: int, bias: bool = True):
        super().__init__()
        # 硬性约束：输入/输出维度必须能被n整除
        assert in_features % n == 0 and out_features % n == 0, \
            "in_features 和 out_features 必须能被 n 整除"

        self.n = n
        self.in_features = in_features
        self.out_features = out_features

        # 可学习参数1：n个 A_i 矩阵，每个形状为 n×n，控制超复数乘法规则
        self.A = nn.Parameter(torch.randn(n, n, n))
        # 可学习参数2：n个 S_i 矩阵，每个形状为 (out/n) × (in/n)，特征变换基底
        self.S = nn.Parameter(torch.randn(n, out_features // n, in_features // n))

        # 偏置项，和标准全连接完全一致
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

        # 初始化：对齐nn.Linear的初始化风格
        nn.init.kaiming_uniform_(self.S, a=5 ** 0.5)
        nn.init.normal_(self.A, mean=0, std=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 核心步骤：用克罗内克积组装完整权重矩阵 H = Σ (A_i ⊗ S_i)
        weight = sum(torch.kron(self.A[i], self.S[i]) for i in range(self.n))
        # 执行线性变换，逻辑和nn.Linear完全一致
        return F.linear(x, weight, self.bias)


# 使用示例
phm_fc_2 = PHMLinear(in_features=d, out_features=q, n=2, bias=True)
out_phm_2 = phm_fc_2(x)
phm_fc_4 = PHMLinear(in_features=d, out_features=q, n=4, bias=True)
out_phm_4 = phm_fc_4(x)
phm_fc_8 = PHMLinear(in_features=d, out_features=q, n=8, bias=True)
out_phm_8 = phm_fc_8(x)
phm_fc_16 = PHMLinear(in_features=d, out_features=q, n=16, bias=True)
out_phm_16 = phm_fc_16(x)
phm_fc_32 = PHMLinear(in_features=d, out_features=q, n=32, bias=True)
out_phm_32 = phm_fc_32(x)
phm_fc_64 = PHMLinear(in_features=d, out_features=q, n=64, bias=True)
out_phm_64 = phm_fc_64(x)

# 统计参数量
phm_param_count_2 = sum(p.numel() for p in phm_fc_2.parameters())
phm_param_count_4 = sum(p.numel() for p in phm_fc_4.parameters())
phm_param_count_8 = sum(p.numel() for p in phm_fc_8.parameters())
phm_param_count_16 = sum(p.numel() for p in phm_fc_16.parameters())
phm_param_count_32 = sum(p.numel() for p in phm_fc_32.parameters())
phm_param_count_64 = sum(p.numel() for p in phm_fc_64.parameters())

print(f"PHM-2 总参数量：{phm_param_count_2}")
print(f"PHM-4 总参数量：{phm_param_count_4}")
print(f"PHM-8 总参数量：{phm_param_count_8}")
print(f"PHM-16 总参数量：{phm_param_count_16}")
print(f"PHM-32 总参数量：{phm_param_count_32}")
print(f"PHM-64 总参数量：{phm_param_count_64}")
# 输出：20992 = 16^3(A矩阵) + (512*512)/16(S矩阵) + 512(偏置)

## 拟合对比实验：标准网络 vs PHM 网络
为了验证 PHM 层在实际训练中的拟合能力、收敛特性与时间开销，构造带噪声的正弦函数回归任务，搭建结构严格对齐的两组网络进行对比实验。
实验设计
- 任务目标：拟合加入高斯噪声的 $\sin(x)$ 非线性函数
- 网络结构：均为 2 层隐藏层的全连接网络；输入层、输出层保持标准全连接，仅将中间高维隐藏层替换为 PHM 层，保证结构公平性
- 对比指标：总参数量、训练损失收敛曲线、单步/总训练耗时、最终拟合效果
- 计时规则：设置预热轮次消除算子冷启动、内存分配的干扰，使用高精度计时器统计纯训练耗时

本段代码完成完整的同步训练过程，并通过可视化呈现损失曲线与拟合效果的差异。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt

# ===================== 1. PHM 全连接层定义 =====================
class PHMLinear(nn.Module):
    def __init__(self, in_features, out_features, n, bias=True):
        super().__init__()
        assert in_features % n == 0 and out_features % n == 0, "维度必须能被n整除"
        self.n = n
        self.in_features = in_features
        self.out_features = out_features

        self.A = nn.Parameter(torch.randn(n, n, n) * 0.01)
        self.S = nn.Parameter(torch.randn(n, out_features//n, in_features//n))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_uniform_(self.S, a=5**0.5)

    def forward(self, x):
        weight = sum(torch.kron(self.A[i], self.S[i]) for i in range(self.n))
        return F.linear(x, weight, self.bias)


# ===================== 2. 生成拟合数据 =====================
torch.manual_seed(42)
x = torch.linspace(-2 * torch.pi, 2 * torch.pi, 2000).unsqueeze(1)
y_true = torch.sin(x)
y = y_true + 0.1 * torch.randn_like(x)


# ===================== 3. 两个对比网络 =====================
class StandardNet(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.layers(x)

class PHMNet(nn.Module):
    def __init__(self, hidden_dim=64, n=4):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            PHMLinear(hidden_dim, hidden_dim, n=n),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.layers(x)


# ===================== 4. 初始化与参数量统计 =====================
hidden_dim = 512
n = 16
epochs = 1500
warmup_epochs = 50  # 预热轮次，不计入计时
lr = 0.005

model_std = StandardNet(hidden_dim=hidden_dim)
model_phm = PHMNet(hidden_dim=hidden_dim, n=n)

count_params = lambda m: sum(p.numel() for p in m.parameters())
param_std = count_params(model_std)
param_phm = count_params(model_phm)

print(f"=== 参数量对比 ===")
print(f"标准网络: {param_std}")
print(f"PHM网络:  {param_phm}")
print(f"压缩比例: {param_phm / param_std:.2%}")
print()

criterion = nn.MSELoss()
optimizer_std = torch.optim.Adam(model_std.parameters(), lr=lr)
optimizer_phm = torch.optim.Adam(model_phm.parameters(), lr=lr)

loss_std_history = []
loss_phm_history = []


# ===================== 5. 预热（不计入正式计时） =====================
print(f"正在进行 {warmup_epochs} 轮预热...")
for _ in range(warmup_epochs):
    # 标准网络预热
    pred = model_std(x)
    loss = criterion(pred, y)
    optimizer_std.zero_grad()
    loss.backward()
    optimizer_std.step()

    # PHM网络预热
    pred = model_phm(x)
    loss = criterion(pred, y)
    optimizer_phm.zero_grad()
    loss.backward()
    optimizer_phm.step()


# ===================== 6. 正式训练 + 精准计时 =====================
print(f"开始正式计时训练，共 {epochs} 轮...")
time_std_total = 0.0
time_phm_total = 0.0

for epoch in range(epochs):
    # ---- 标准网络单步计时 ----
    t0 = time.perf_counter()
    pred_std = model_std(x)
    loss_std = criterion(pred_std, y)
    optimizer_std.zero_grad()
    loss_std.backward()
    optimizer_std.step()
    time_std_total += time.perf_counter() - t0
    loss_std_history.append(loss_std.item())

    # ---- PHM网络单步计时 ----
    t0 = time.perf_counter()
    pred_phm = model_phm(x)
    loss_phm = criterion(pred_phm, y)
    optimizer_phm.zero_grad()
    loss_phm.backward()
    optimizer_phm.step()
    time_phm_total += time.perf_counter() - t0
    loss_phm_history.append(loss_phm.item())

    if (epoch + 1) % 300 == 0:
        print(f"Epoch {epoch+1:4d} | 标准MSE: {loss_std.item():.6f} | PHM MSE: {loss_phm.item():.6f}")


# ===================== 7. 时间统计输出 =====================
avg_std = time_std_total / epochs * 1000  # 毫秒
avg_phm = time_phm_total / epochs * 1000

print()
print(f"=== 训练时间对比（共 {epochs} 轮） ===")
print(f"标准网络总耗时: {time_std_total:.3f} s | 单步平均: {avg_std:.3f} ms")
print(f"PHM网络总耗时:  {time_phm_total:.3f} s | 单步平均: {avg_phm:.3f} ms")
print(f"PHM相对速度:    {time_phm_total / time_std_total:.2f} 倍（标准网络为 1.0）")
print()


# ===================== 8. 可视化对比 =====================
model_std.eval()
model_phm.eval()
with torch.no_grad():
    pred_std_final = model_std(x)
    pred_phm_final = model_phm(x)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_std_history, label=f'Standard FC ({param_std} params)', alpha=0.8)
plt.plot(loss_phm_history, label=f'PHM n={n} ({param_phm} params)', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(x.numpy(), y.numpy(), s=5, c='gray', alpha=0.3, label='Noisy Data')
plt.plot(x.numpy(), pred_std_final.numpy(), 'b-', linewidth=2, label='Standard FC')
plt.plot(x.numpy(), pred_phm_final.numpy(), 'r--', linewidth=2, label='PHM')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Fitting Result')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()